# Time series fundamentals

Everything so far assumed rows were **independent** and interchangeable. Time
series break that: the *order* carries information, and two assumptions from the
[Model Evaluation](../01d-evaluation/cross-validation.ipynb) chapter now fail.

- **Random train/test splitting leaks the future** into training.
- **K-fold needs a time-aware variant** (rolling-origin / expanding window).

This is "a different data shape" — we treat it on its own terms.

In [ ]:
// A synthetic daily series: upward trend + weekly (period-7) seasonality.
let series: Vec<f64> = (0..56).map(|t| {
    let trend = 10.0 + 0.2 * t as f64;
    let weekly = 3.0 * ((t as f64) * 2.0 * std::f64::consts::PI / 7.0).sin();
    trend + weekly
}).collect();
let split = (series.len() as f64 * 0.8) as usize;

// Chronological split: train on the EARLIER data, test on the later.
{
    let (train, test) = series.split_at(split);
    println!("series = {} points; train = {}, test = {}", series.len(), train.len(), test.len());
    println!("split at index {} — never shuffle across this boundary", split);
}

## Rolling-origin cross-validation

Instead of random folds, grow the training window forward in time and always
test on the *next* chunk. Each fold trains only on data that precedes its test
set — no leakage:

In [ ]:
{
    let initial = 28;   // first training window
    let horizon = 7;    // test one week ahead each fold
    let mut origin = initial;
    let mut fold = 1;
    while origin + horizon <= series.len() {
        println!("fold {}: train [0..{}]  ->  test [{}..{}]", fold, origin, origin, origin + horizon);
        origin += horizon;
        fold += 1;
    }
}

## Lag features & rolling statistics

To let an ordinary regressor use the past, engineer **lag** columns (the value
*k* steps ago) and **rolling** statistics (a moving average). This ties back to
the [ETL chapter's](../01c-etl/data-preparation.ipynb) feature engineering, now
with time awareness:

In [ ]:
{
    // For each day t (from 7 on), build [value, lag-1, lag-7, rolling-mean-3].
    println!("{:>3}  {:>7}  {:>7}  {:>7}  {:>10}", "t", "value", "lag1", "lag7", "roll_mean3");
    for t in 7..12 {
        let value = series[t];
        let lag1 = series[t - 1];
        let lag7 = series[t - 7];
        let roll_mean3 = (series[t - 1] + series[t - 2] + series[t - 3]) / 3.0;
        println!("{:>3}  {:>7.2}  {:>7.2}  {:>7.2}  {:>10.2}", t, value, lag1, lag7, roll_mean3);
    }
}

Those engineered columns turn a forecasting problem into an ordinary supervised
one. Next: [forecasting](forecasting.ipynb) — a naive baseline, a real model
(`augurs` MSTL), and time-series accuracy metrics.